# 01 — Data Cleaning

**Project:** Telco Customer Churn Analysis

This notebook loads the raw Telco Customer Churn export, walks through the cleaning issues found in the data, and writes an analysis-ready CSV to `data/processed/`.

The cleaning logic itself lives in [`scripts/data_cleaning.py`](../scripts/data_cleaning.py) so it can be reused by the notebook, the SQL load step, and any future automation — this notebook calls it and inspects the results.

## 1. Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "scripts"))

from data_cleaning import load_raw_data, clean_customer_churn, RAW_PATH, PROCESSED_PATH

pd.set_option("display.max_columns", None)

## 2. Load raw data

In [2]:
raw_df = load_raw_data()
print(f"Rows: {raw_df.shape[0]}, Columns: {raw_df.shape[1]}")
raw_df.head()

Rows: 7043, Columns: 21


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 3. Data quality checks

Before cleaning, confirm what's actually wrong with the raw export.

In [3]:
print("Missing values per column:")
print(raw_df.isna().sum()[raw_df.isna().sum() > 0])

print("\nTotalCharges dtype:", raw_df["TotalCharges"].dtype)
blank_total_charges = raw_df[raw_df["TotalCharges"].str.strip() == ""]
print(f"Rows with blank TotalCharges: {len(blank_total_charges)}")
blank_total_charges[["customerID", "tenure", "MonthlyCharges", "TotalCharges"]]

Missing values per column:
Series([], dtype: int64)

TotalCharges dtype: object
Rows with blank TotalCharges: 11


,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,
753,3115-CZMZD,0,20.25,
936,5709-LVOEQ,0,80.85,
1082,4367-NUYAO,0,25.75,
1340,1371-DWPAZ,0,56.05,
3331,7644-OMVMY,0,19.85,
3826,3213-VVOLG,0,25.35,
4380,2520-SGTTA,0,20.00,
5218,2923-ARZLG,0,19.70,
6670,4075-WKNIU,0,73.35,


**Finding:** 11 customers have a blank `TotalCharges` value. All of them have `tenure = 0`, meaning they are brand-new customers who haven't been billed yet — not a data entry error. Since there's no reliable way to impute a first bill and it's a tiny fraction of the dataset (0.16%), these rows are dropped rather than guessed at.

Also note `SeniorCitizen` is stored as `0`/`1` instead of `Yes`/`No` like every other flag column — inconsistent encoding that gets standardized below.

## 4. Apply cleaning pipeline

In [4]:
clean_df = clean_customer_churn(raw_df)
print(f"Rows before: {len(raw_df)}, after: {len(clean_df)} (dropped {len(raw_df) - len(clean_df)})")
clean_df.head()

Rows before: 7043, after: 7032 (dropped 11)


,customer_id,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,online_backup,device_protection,tech_support,streaming_t_v,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn,churn_flag,tenure_group
0,7590-VHVEG,Female,No,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0,0-12 mo
1,5575-GNVDE,Male,No,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No,0,25-48 mo
2,3668-QPYBK,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,0-12 mo
3,7795-CFOCW,Male,No,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0,25-48 mo
4,9237-HQITU,Female,No,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,0-12 mo


In [5]:
assert clean_df["total_charges"].isna().sum() == 0
assert clean_df["customer_id"].is_unique
assert set(clean_df["churn_flag"].unique()) == {0, 1}
print("All cleaning checks passed.")

All cleaning checks passed.


## 5. Save processed dataset

In [6]:
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
clean_df.to_csv(PROCESSED_PATH, index=False)
print(f"Saved cleaned dataset to {PROCESSED_PATH.relative_to(PROJECT_ROOT)}")

Saved cleaned dataset to data\processed\telco_customer_churn_clean.csv
